# FASE 2 — Limpeza e Pré-processamento dos Dados

Este notebook documenta o processo de limpeza e preparação do dataset da Netflix, corrigindo inconsistências identificadas na Fase 1 (Diagnóstico).

In [ ]:
import pandas as pd
import numpy as np
import os

# Configurações de exibição
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados Brutos

In [ ]:
df = pd.read_csv('../dados_brutos/netflix_raw.csv')
print(f"Formato original: {df.shape}")
df.head()

## 2. Tratamento de Missing Values (Placeholders)

Substituindo textos que indicam valores ausentes por `NaN` real do Pandas.

In [ ]:
placeholders = ['Not Given', '???', 'NULL', 'N/A', '', 'Unknown']
df.replace(placeholders, np.nan, inplace=True)
print(f"Nulos após tratamento de placeholders:\n{df.isnull().sum()}")

## 3. Padronização Textual

Removendo espaços extras (trim) e garantindo que strings vazias ou 'nan' sejam tratadas como nulas.

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col] == 'nan', col] = np.nan

## 4. Correção Estrutural (Colunas Deslocadas)

Corrigindo casos onde a classificação indicativa (rating) foi parar na coluna de tipo (`type`).

In [ ]:
valid_types = ['Movie', 'TV Show']
mask_shifted = df['type'].notna() & ~df['type'].str.title().isin(valid_types)

# Mover rating do 'type' para a coluna 'rating' se estiver nulo
df.loc[mask_shifted & df['rating'].isna(), 'rating'] = df.loc[mask_shifted, 'type']

# Inferir o tipo correto com base na duração
def infer_type(duration):
    if pd.isna(duration): return np.nan
    d = str(duration).lower()
    return 'TV Show' if 'season' in d else 'Movie'

df.loc[mask_shifted, 'type'] = df.loc[mask_shifted, 'duration'].apply(infer_type)

# Padronizar 'type' de forma robusta
def standardize_type(val):
    if pd.isna(val): return val
    v = str(val).strip().lower()
    if 'movie' in v: return 'Movie'
    if 'tv show' in v or 'tv' in v: return 'TV Show'
    return val

df['type'] = df['type'].apply(standardize_type)
df.loc[~df['type'].isin(valid_types), 'type'] = np.nan
print(f"Tipos únicos após correção: {df['type'].unique()}")

## 5. Tratamento de Duplicados

Removendo registros exatos e IDs repetidos.

In [ ]:
before = len(df)
df.drop_duplicates(subset=[c for c in df.columns if c != 'show_id'], inplace=True)
df.drop_duplicates(subset='show_id', inplace=True)
after = len(df)
print(f"Registros removidos: {before - after}")

## 6. Padronização de Unidades e Tipos

Padronizando a coluna `duration` e convertendo datas.

In [ ]:
def standardize_duration(d):
    if pd.isna(d): return d
    d = str(d).lower()
    num = ''.join(filter(str.isdigit, d))
    if 'min' in d or 'minute' in d or 'm' in d:
        return f"{num} min"
    if 'season' in d:
        return f"{num} Season" if num == "1" else f"{num} Seasons"
    return d

df['duration'] = df['duration'].apply(standardize_duration)
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
df.info()

## 7. Verificação Lógica

Identificando inconsistências temporais (data de adição anterior ao lançamento).

In [ ]:
invalid = df[df['date_added'].dt.year < df['release_year']]
print(f"Registros inconsistentes encontrados: {len(invalid)}")

## 8. Salvando o Resultado

Exportando para a pasta de dados intermediários.

In [ ]:
os.makedirs('../dados_intermediarios', exist_ok=True)
df.to_csv('../dados_intermediarios/netflix_clean.csv', index=False)
print("Arquivo salvo com sucesso!")